In [ ]:
# Plot gradient norms evolution over training
if grad_stats_dict:
    num_optimizers = len(grad_stats_dict)
    opt_types = list(grad_stats_dict.keys())
    
    # Create figure with 3 rows (spectral norm, frobenius norm, trace)
    fig, axes = plt.subplots(3, 1, figsize=(16, 14))
    
    # Colors for different optimizer types
    opt_colors = plt.cm.tab10(np.linspace(0, 1, num_optimizers))
    
    # Extract data for plotting
    for idx, opt_type in enumerate(opt_types):
        grad_data = grad_stats_dict[opt_type]['grad_stats']
        
        # Extract steps and statistics
        steps = []
        spectral_norms = []
        frobenius_norms = []
        traces = []
        
        for step, stats in grad_data:
            if stats:  # Check if stats dict is not empty
                steps.append(step)
                spectral_norms.append(stats.get('spectral_norm', np.nan))
                frobenius_norms.append(stats.get('frobenius_norm', np.nan))
                traces.append(stats.get('trace', np.nan))
        
        # Convert to numpy arrays
        steps = np.array(steps)
        spectral_norms = np.array(spectral_norms)
        frobenius_norms = np.array(frobenius_norms)
        traces = np.array(traces)
        
        # Plot spectral norm
        axes[0].plot(steps, spectral_norms, 
                     label=f'{opt_type} (Acc: {grad_stats_dict[opt_type]["test_acc"]:.4f})',
                     color=opt_colors[idx], linewidth=2.5, marker='o', markersize=6,
                     markevery=max(1, len(steps)//15))
        
        # Plot Frobenius norm
        axes[1].plot(steps, frobenius_norms,
                     label=f'{opt_type} (Acc: {grad_stats_dict[opt_type]["test_acc"]:.4f})',
                     color=opt_colors[idx], linewidth=2.5, marker='s', markersize=6,
                     markevery=max(1, len(steps)//15))
        
        # Plot trace
        axes[2].plot(steps, traces,
                     label=f'{opt_type} (Acc: {grad_stats_dict[opt_type]["test_acc"]:.4f})',
                     color=opt_colors[idx], linewidth=2.5, marker='^', markersize=6,
                     markevery=max(1, len(steps)//15))
    
    # Configure spectral norm plot
    axes[0].set_xlabel('Training Step', fontweight='bold', fontsize=13)
    axes[0].set_ylabel('Spectral Norm ||∇||₂', fontweight='bold', fontsize=13)
    axes[0].set_title('Gradient Spectral Norm Evolution (Largest Singular Value)',
                      fontweight='bold', fontsize=15)
    axes[0].legend(loc='best', fontsize=11, framealpha=0.9)
    axes[0].grid(True, alpha=0.3)
    axes[0].set_yscale('log')
    
    # Configure Frobenius norm plot
    axes[1].set_xlabel('Training Step', fontweight='bold', fontsize=13)
    axes[1].set_ylabel('Frobenius Norm ||∇||_F', fontweight='bold', fontsize=13)
    axes[1].set_title('Gradient Frobenius Norm Evolution',
                      fontweight='bold', fontsize=15)
    axes[1].legend(loc='best', fontsize=11, framealpha=0.9)
    axes[1].grid(True, alpha=0.3)
    axes[1].set_yscale('log')
    
    # Configure trace plot
    axes[2].set_xlabel('Training Step', fontweight='bold', fontsize=13)
    axes[2].set_ylabel('Trace Tr(∇ᵀ∇)', fontweight='bold', fontsize=13)
    axes[2].set_title('Gradient Gram Matrix Trace Evolution (Sum of Squared Singular Values)',
                      fontweight='bold', fontsize=15)
    axes[2].legend(loc='best', fontsize=11, framealpha=0.9)
    axes[2].grid(True, alpha=0.3)
    axes[2].set_yscale('log')
    
    plt.tight_layout()
    plt.savefig(os.path.join(LOG_DIR, 'gradient_norms_evolution.png'), dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\nGradient norms plot saved to: {os.path.join(LOG_DIR, 'gradient_norms_evolution.png')}")
else:
    print("Skipping gradient statistics plots - no data available")

In [ ]:
# Load gradient statistics for best optimizer of each type
def load_gradient_statistics(config_str, log_dir):
    """Load gradient statistics for a given optimizer config."""
    safe_name = config_str.replace(".", "_").replace("/", "_")
    grad_stats_path = os.path.join(log_dir, f"gradient_stats_{safe_name}.pkl")
    
    if not os.path.exists(grad_stats_path):
        print(f"Warning: Gradient statistics not found for {config_str}")
        return None
    
    with open(grad_stats_path, 'rb') as f:
        grad_stats_data = pickle.load(f)
    
    return grad_stats_data

# Load gradient statistics for best optimizers (already defined in best_per_type)
grad_stats_dict = {}
for opt_type, best_config in best_per_type.items():
    grad_stats = load_gradient_statistics(best_config['config'], LOG_DIR)
    if grad_stats:
        grad_stats_dict[opt_type] = {
            'config': best_config['config'],
            'grad_stats': grad_stats,
            'test_acc': best_config['final_test_acc']
        }

if grad_stats_dict:
    print(f"Loaded gradient statistics for {len(grad_stats_dict)} optimizer types")
    for opt_type, data in grad_stats_dict.items():
        print(f"  {opt_type}: {len(data['grad_stats'])} snapshots")
        if len(data['grad_stats']) > 0:
            _, first_stats = data['grad_stats'][0]
            if 'param_name' in first_stats:
                print(f"    Tracking parameter: {first_stats['param_name']}")
else:
    print("No gradient statistics found. This feature requires CifarNet architecture.")

In [ ]:
# Create static comparison at final step
fig, axes = plt.subplots(1, num_optimizers, figsize=(7*num_optimizers, 6))

if num_optimizers == 1:
    axes = [axes]

for idx, opt_type in enumerate(opt_types):
    ax = axes[idx]
    
    # Get final singular values
    sv_data_list = sv_data_dict[opt_type]['sv_data']
    final_step, final_sv_dict = sv_data_list[-1]
    
    # Collect all singular values
    all_svs = []
    for layer_name, svs in final_sv_dict.items():
        if isinstance(svs, (list, np.ndarray)):
            all_svs.extend(svs)
    
    all_svs = np.array(all_svs)
    all_svs = all_svs[all_svs > 0]
    
    # Create histogram
    bins = np.logspace(np.log10(all_svs.min()), np.log10(all_svs.max()), 50)
    counts, edges = np.histogram(all_svs, bins=bins)
    
    ax.bar(edges[:-1], counts, width=np.diff(edges),
           color=opt_colors[idx], alpha=0.7, edgecolor='black', linewidth=1)
    
    ax.set_xlabel('Singular Value (log scale)', fontweight='bold', fontsize=12)
    ax.set_ylabel('Frequency (log scale)', fontweight='bold', fontsize=12)
    ax.set_title(f'{opt_type} (Step {final_step})\nAcc: {sv_data_dict[opt_type]["test_acc"]:.4f}',
                 fontweight='bold', fontsize=13)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3)
    
    # Add statistics
    stats_text = f'Min: {all_svs.min():.2e}\nMax: {all_svs.max():.2e}\nMean: {all_svs.mean():.2e}\nStd: {all_svs.std():.2e}'
    ax.text(0.98, 0.98, stats_text, transform=ax.transAxes,
            fontsize=10, verticalalignment='top', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7))

fig.suptitle('Final Singular Value Distribution - Best Optimizers per Type',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(LOG_DIR, 'final_singular_values_comparison.png'), dpi=300, bbox_inches='tight')
plt.show()

print(f"\nStatic comparison saved to: {os.path.join(LOG_DIR, 'final_singular_values_comparison.png')}")

## Gradient Statistics Analysis
### Evolution of Gradient Properties During Training

## Static Singular Value Analysis
### Comparison at Final Step

In [ ]:
# Create animated histogram visualization
print("\nCreating animation... (this may take a moment)")

# Determine common time steps across all optimizers
all_steps = set()
for opt_type, data in sv_data_dict.items():
    for step, sv_dict in data['sv_data']:
        all_steps.add(step)
common_steps = sorted(list(all_steps))

print(f"Total frames: {len(common_steps)}")

# Set up the figure
num_optimizers = len(sv_data_dict)
fig, axes = plt.subplots(1, num_optimizers, figsize=(7*num_optimizers, 6))

if num_optimizers == 1:
    axes = [axes]

# Colors for different optimizer types
opt_colors = plt.cm.tab10(np.linspace(0, 1, num_optimizers))

# Initialize plots
opt_types = list(sv_data_dict.keys())

def update(frame_idx):
    """Update function for animation."""
    step = common_steps[frame_idx]
    
    for idx, opt_type in enumerate(opt_types):
        ax = axes[idx]
        
        # Find singular values for this step
        sv_data_list = sv_data_dict[opt_type]['sv_data']
        sv_dict = None
        for s, sv in sv_data_list:
            if s == step:
                sv_dict = sv
                break
        
        if sv_dict is None:
            continue
        
        # Collect all singular values across all layers
        all_svs = []
        for layer_name, svs in sv_dict.items():
            if isinstance(svs, (list, np.ndarray)):
                all_svs.extend(svs)
        
        if len(all_svs) == 0:
            continue
        
        all_svs = np.array(all_svs)
        all_svs = all_svs[all_svs > 0]  # Remove zeros for log scale
        
        # Clear previous bars
        ax.clear()
        
        # Create histogram
        bins = np.logspace(np.log10(all_svs.min()), np.log10(all_svs.max()), 50)
        counts, edges = np.histogram(all_svs, bins=bins)
        
        # Plot histogram
        ax.bar(edges[:-1], counts, width=np.diff(edges),
               color=opt_colors[idx], alpha=0.7, edgecolor='black', linewidth=0.5)
        
        # Set labels and title
        ax.set_xlabel('Singular Value (log scale)', fontweight='bold', fontsize=12)
        ax.set_ylabel('Frequency (log scale)', fontweight='bold', fontsize=12)
        ax.set_title(f'{opt_type}\nAcc: {sv_data_dict[opt_type]["test_acc"]:.4f}',
                     fontweight='bold', fontsize=13)
        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.grid(True, alpha=0.3)
        
        # Add step annotation
        ax.text(0.02, 0.98, f'Step: {step}', transform=ax.transAxes,
                fontsize=11, verticalalignment='top', fontweight='bold',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
        
        # Add statistics
        stats_text = f'Min: {all_svs.min():.2e}\nMax: {all_svs.max():.2e}\nMean: {all_svs.mean():.2e}'
        ax.text(0.98, 0.98, stats_text, transform=ax.transAxes,
                fontsize=9, verticalalignment='top', horizontalalignment='right',
                bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.6))
    
    fig.suptitle(f'Singular Value Distribution Evolution - Step {step}',
                 fontsize=16, fontweight='bold', y=0.98)
    
    return []

# Create animation
print("Rendering animation frames...")
anim = FuncAnimation(fig, update, frames=len(common_steps),
                     interval=200, blit=False, repeat=True)

# Save as HTML5 video for notebook display
plt.tight_layout()
html_anim = HTML(anim.to_jshtml())

# Also save as GIF
gif_path = os.path.join(LOG_DIR, 'singular_values_evolution.gif')
print(f"Saving animation to {gif_path}...")
anim.save(gif_path, writer='pillow', fps=5, dpi=100)
print(f"Animation saved!")

# Display in notebook
html_anim

In [ ]:
# Load singular value data for best optimizer of each type
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

def load_singular_values(config_str, log_dir):
    """Load singular values for a given optimizer config."""
    safe_name = config_str.replace(".", "_").replace("/", "_")
    sv_path = os.path.join(log_dir, f"singular_values_{safe_name}.pkl")
    
    if not os.path.exists(sv_path):
        print(f"Warning: Singular values not found for {config_str}")
        return None
    
    with open(sv_path, 'rb') as f:
        sv_data = pickle.load(f)
    
    return sv_data

# Get best optimizer for each type
best_per_type = {}
for opt_type, top3_list in top3_metrics.items():
    # Best is the first one (already sorted by test_acc)
    best_per_type[opt_type] = top3_list[0]

# Load singular values for best optimizers
sv_data_dict = {}
for opt_type, best_config in best_per_type.items():
    sv_data = load_singular_values(best_config['config'], LOG_DIR)
    if sv_data:
        sv_data_dict[opt_type] = {
            'config': best_config['config'],
            'sv_data': sv_data,
            'test_acc': best_config['final_test_acc']
        }

print(f"Loaded singular values for {len(sv_data_dict)} optimizer types")
for opt_type, data in sv_data_dict.items():
    print(f"  {opt_type}: {len(data['sv_data'])} snapshots")

# Training Results Analysis
## Top 3 Optimizers per Type by Final Test Accuracy

In [ ]:
import os
import json
import pickle
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict

# Set up plotting style
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (15, 10)
plt.rcParams['font.size'] = 12

In [ ]:
# ========== CONFIGURATION ==========
# Set your log directory path here
LOG_DIR = 'logs/HybridMR_72opts_a3f2c1d8'  # Replace with your actual log directory

# Verify directory exists
if not os.path.exists(LOG_DIR):
    raise FileNotFoundError(f"Log directory not found: {LOG_DIR}")

print(f"Analyzing results from: {LOG_DIR}")

## Load Configuration and Results

In [ ]:
# Load config.pt
config_path = os.path.join(LOG_DIR, 'config.pt')
config_data = torch.load(config_path)

optimizer_configs = config_data['optimizer_configs']
print(f"Total optimizers: {len(optimizer_configs)}")
print(f"\nFirst few optimizer configs:")
for i, cfg in enumerate(optimizer_configs[:3]):
    print(f"  {i+1}. {cfg}")

In [ ]:
# Extract final test accuracies and group by optimizer type
results = []

for opt_config_str in optimizer_configs:
    # Get optimizer type (first part before underscore)
    opt_type = opt_config_str.split('_')[0]
    
    # Get final test accuracy from config_data
    test_acc_key = f'test_acc_{opt_config_str}'
    if test_acc_key in config_data:
        final_test_acc = config_data[test_acc_key]
    else:
        final_test_acc = 0.0
    
    results.append({
        'config': opt_config_str,
        'type': opt_type,
        'final_test_acc': final_test_acc
    })

print(f"\nLoaded {len(results)} optimizer results")

## Group by Optimizer Type and Find Top 3

In [ ]:
# Group by optimizer type
grouped = defaultdict(list)
for result in results:
    grouped[result['type']].append(result)

# Sort each group by final test accuracy and get top 3
top3_per_type = {}
for opt_type, opt_results in grouped.items():
    sorted_results = sorted(opt_results, key=lambda x: x['final_test_acc'], reverse=True)
    top3_per_type[opt_type] = sorted_results[:3]

# Display top 3 per type
print("Top 3 Optimizers by Type:\n")
for opt_type, top3 in top3_per_type.items():
    print(f"\n{'='*80}")
    print(f"Optimizer Type: {opt_type}")
    print(f"{'='*80}")
    for i, result in enumerate(top3, 1):
        print(f"\n  {i}. Test Acc: {result['final_test_acc']:.4f}")
        print(f"     Config: {result['config']}")

## Load Training Metrics and Plot

In [ ]:
def load_metrics(config_str, log_dir):
    """Load metrics for a given optimizer config."""
    safe_name = config_str.replace(".", "_").replace("/", "_")
    metrics_path = os.path.join(log_dir, f"metrics_{safe_name}.json")
    
    if not os.path.exists(metrics_path):
        print(f"Warning: Metrics not found for {config_str}")
        return None
    
    with open(metrics_path, 'r') as f:
        metrics = json.load(f)
    
    return metrics

# Load metrics for all top 3 optimizers
top3_metrics = {}
for opt_type, top3 in top3_per_type.items():
    top3_metrics[opt_type] = []
    for result in top3:
        metrics = load_metrics(result['config'], LOG_DIR)
        if metrics:
            top3_metrics[opt_type].append({
                'config': result['config'],
                'metrics': metrics,
                'final_test_acc': result['final_test_acc']
            })

print("Loaded metrics for all top performers")

In [ ]:
# Plot training and test losses for each optimizer type
num_types = len(top3_metrics)
fig, axes = plt.subplots(num_types, 2, figsize=(20, 6*num_types))

if num_types == 1:
    axes = axes.reshape(1, -1)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c']  # Blue, Orange, Green for top 3
markers = ['o', 's', '^']  # Circle, Square, Triangle

for idx, (opt_type, top3_list) in enumerate(top3_metrics.items()):
    ax_train = axes[idx, 0]
    ax_test = axes[idx, 1]
    
    # Plot training loss
    for i, item in enumerate(top3_list):
        metrics = item['metrics']
        epochs = [m['epoch'] for m in metrics]
        train_loss = [m['train_loss'] for m in metrics]
        
        label = f"{item['config'][:50]}... (Acc: {item['final_test_acc']:.4f})"
        ax_train.plot(epochs, train_loss, 
                     label=label, 
                     color=colors[i], 
                     marker=markers[i],
                     markevery=max(1, len(epochs)//10),
                     linewidth=2,
                     markersize=8)
    
    ax_train.set_xlabel('Epoch', fontsize=14, fontweight='bold')
    ax_train.set_ylabel('Training Loss', fontsize=14, fontweight='bold')
    ax_train.set_title(f'{opt_type} - Training Loss (Top 3)', fontsize=16, fontweight='bold')
    ax_train.legend(loc='best', fontsize=10)
    ax_train.grid(True, alpha=0.3)
    
    # Plot test loss
    for i, item in enumerate(top3_list):
        metrics = item['metrics']
        epochs = [m['epoch'] for m in metrics]
        test_loss = [m['test_loss'] for m in metrics]
        
        label = f"{item['config'][:50]}... (Acc: {item['final_test_acc']:.4f})"
        ax_test.plot(epochs, test_loss, 
                    label=label, 
                    color=colors[i], 
                    marker=markers[i],
                    markevery=max(1, len(epochs)//10),
                    linewidth=2,
                    markersize=8)
    
    ax_test.set_xlabel('Epoch', fontsize=14, fontweight='bold')
    ax_test.set_ylabel('Test Loss', fontsize=14, fontweight='bold')
    ax_test.set_title(f'{opt_type} - Test Loss (Top 3)', fontsize=16, fontweight='bold')
    ax_test.legend(loc='best', fontsize=10)
    ax_test.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(LOG_DIR, 'top3_losses_comparison.png'), dpi=300, bbox_inches='tight')
plt.show()

print(f"\nPlot saved to: {os.path.join(LOG_DIR, 'top3_losses_comparison.png')}")

## Plot Test Accuracy Curves

In [ ]:
# Plot test accuracy for each optimizer type
fig, axes = plt.subplots(num_types, 1, figsize=(20, 6*num_types))

if num_types == 1:
    axes = [axes]

for idx, (opt_type, top3_list) in enumerate(top3_metrics.items()):
    ax = axes[idx]
    
    for i, item in enumerate(top3_list):
        metrics = item['metrics']
        epochs = [m['epoch'] for m in metrics]
        test_acc = [m['test_acc'] for m in metrics]
        
        label = f"{item['config'][:50]}... (Final: {item['final_test_acc']:.4f})"
        ax.plot(epochs, test_acc, 
               label=label, 
               color=colors[i], 
               marker=markers[i],
               markevery=max(1, len(epochs)//10),
               linewidth=2,
               markersize=8)
    
    ax.set_xlabel('Epoch', fontsize=14, fontweight='bold')
    ax.set_ylabel('Test Accuracy', fontsize=14, fontweight='bold')
    ax.set_title(f'{opt_type} - Test Accuracy (Top 3)', fontsize=16, fontweight='bold')
    ax.legend(loc='best', fontsize=10)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(LOG_DIR, 'top3_accuracy_comparison.png'), dpi=300, bbox_inches='tight')
plt.show()

print(f"\nPlot saved to: {os.path.join(LOG_DIR, 'top3_accuracy_comparison.png')}")

## Summary Statistics

In [ ]:
# Create summary table
import pandas as pd

summary_data = []
for opt_type, top3_list in top3_metrics.items():
    for rank, item in enumerate(top3_list, 1):
        metrics = item['metrics']
        final_metrics = metrics[-1]
        
        summary_data.append({
            'Optimizer Type': opt_type,
            'Rank': rank,
            'Config': item['config'],
            'Final Test Acc': f"{item['final_test_acc']:.4f}",
            'Final Test Loss': f"{final_metrics['test_loss']:.4f}",
            'Final Train Acc': f"{final_metrics['train_acc']:.4f}",
            'Final Train Loss': f"{final_metrics['train_loss']:.4f}",
            'Final LR': f"{final_metrics['lr']:.6f}",
        })

df_summary = pd.DataFrame(summary_data)
print("\n" + "="*100)
print("SUMMARY: Top 3 Optimizers per Type")
print("="*100)
print(df_summary.to_string(index=False))

# Save summary to CSV
csv_path = os.path.join(LOG_DIR, 'top3_summary.csv')
df_summary.to_csv(csv_path, index=False)
print(f"\nSummary saved to: {csv_path}")

## Overall Best Optimizer

In [ ]:
# Find overall best optimizer across all types
all_top_configs = []
for opt_type, top3_list in top3_metrics.items():
    all_top_configs.extend(top3_list)

best_overall = max(all_top_configs, key=lambda x: x['final_test_acc'])

print("\n" + "="*100)
print("OVERALL BEST OPTIMIZER")
print("="*100)
print(f"\nConfig: {best_overall['config']}")
print(f"Final Test Accuracy: {best_overall['final_test_acc']:.4f}")
print(f"\nFull Training Curve:")

# Plot best optimizer's full training curve
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

metrics = best_overall['metrics']
epochs = [m['epoch'] for m in metrics]

# Train/Test Loss
axes[0].plot(epochs, [m['train_loss'] for m in metrics], label='Train Loss', linewidth=2, marker='o')
axes[0].plot(epochs, [m['test_loss'] for m in metrics], label='Test Loss', linewidth=2, marker='s')
axes[0].set_xlabel('Epoch', fontweight='bold')
axes[0].set_ylabel('Loss', fontweight='bold')
axes[0].set_title('Loss Curves', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Train/Test Accuracy
axes[1].plot(epochs, [m['train_acc'] for m in metrics], label='Train Acc', linewidth=2, marker='o')
axes[1].plot(epochs, [m['test_acc'] for m in metrics], label='Test Acc', linewidth=2, marker='s')
axes[1].set_xlabel('Epoch', fontweight='bold')
axes[1].set_ylabel('Accuracy', fontweight='bold')
axes[1].set_title('Accuracy Curves', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Learning Rate Schedule
axes[2].plot(epochs, [m['lr'] for m in metrics], linewidth=2, marker='o', color='purple')
axes[2].set_xlabel('Epoch', fontweight='bold')
axes[2].set_ylabel('Learning Rate', fontweight='bold')
axes[2].set_title('Learning Rate Schedule', fontweight='bold')
axes[2].grid(True, alpha=0.3)

plt.suptitle(f"Best Overall: {best_overall['config'][:80]}", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(LOG_DIR, 'best_optimizer_curves.png'), dpi=300, bbox_inches='tight')
plt.show()

print(f"\nBest optimizer plot saved to: {os.path.join(LOG_DIR, 'best_optimizer_curves.png')}")

## Compare All Optimizer Types (Average Performance)

In [ ]:
# Calculate average performance per optimizer type
type_performance = {}
for opt_type, opt_results in grouped.items():
    test_accs = [r['final_test_acc'] for r in opt_results]
    type_performance[opt_type] = {
        'mean': np.mean(test_accs),
        'std': np.std(test_accs),
        'max': np.max(test_accs),
        'min': np.min(test_accs),
        'count': len(test_accs)
    }

# Create bar plot
fig, ax = plt.subplots(figsize=(12, 6))

types = list(type_performance.keys())
means = [type_performance[t]['mean'] for t in types]
stds = [type_performance[t]['std'] for t in types]
maxs = [type_performance[t]['max'] for t in types]

x = np.arange(len(types))
width = 0.35

bars1 = ax.bar(x - width/2, means, width, label='Mean Test Acc', yerr=stds, capsize=5)
bars2 = ax.bar(x + width/2, maxs, width, label='Max Test Acc')

ax.set_xlabel('Optimizer Type', fontweight='bold', fontsize=12)
ax.set_ylabel('Test Accuracy', fontweight='bold', fontsize=12)
ax.set_title('Average Performance by Optimizer Type', fontweight='bold', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(types, rotation=45, ha='right')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar in bars1:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.4f}', ha='center', va='bottom', fontsize=9)

for bar in bars2:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.4f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(LOG_DIR, 'optimizer_type_comparison.png'), dpi=300, bbox_inches='tight')
plt.show()

# Print statistics
print("\nPerformance Statistics by Optimizer Type:")
print("="*80)
for opt_type, perf in type_performance.items():
    print(f"\n{opt_type}:")
    print(f"  Count: {perf['count']}")
    print(f"  Mean ± Std: {perf['mean']:.4f} ± {perf['std']:.4f}")
    print(f"  Max: {perf['max']:.4f}")
    print(f"  Min: {perf['min']:.4f}")

## Animated Singular Value Histograms
### Best Optimizer per Type - Evolution During Training